In [1]:
!pip install -q groq scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.1 MB/s eta 0:00:00


In [22]:
import pandas as pd
import json
import time
from sklearn.metrics import cohen_kappa_score, accuracy_score
from groq import Groq
from google.colab import userdata

# Initialize the Groq Client
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# 1. Create our Time-Series Customer Chat Dataset
data = [
    {"customer_id": "C-101", "date": "2026-04-01", "chat_log": "Hi, I'm having trouble exporting my report. It keeps loading.", "human_baseline_risk": "Low"},
    {"customer_id": "C-101", "date": "2026-04-10", "chat_log": "This is the second time I'm asking. The export is still broken. I need this for a meeting.", "human_baseline_risk": "Medium"},
    {"customer_id": "C-101", "date": "2026-04-18", "chat_log": "Your system is completely useless. If this isn't fixed today, I am moving to your competitor.", "human_baseline_risk": "High"},
    {"customer_id": "C-102", "date": "2026-04-15", "chat_log": "Just wanted to say the new UI update looks great! How do I change my avatar?", "human_baseline_risk": "Low"},
    {"customer_id": "C-103", "date": "2026-04-19", "chat_log": "Billing error on my account. I was charged twice. Please refund immediately.", "human_baseline_risk": "Medium"}
]

df = pd.DataFrame(data)
print("Dataset Loaded. Shape:", df.shape)

Dataset Loaded. Shape: (5, 4)


In [24]:
import json
import time

# --- STEP 2 & 3 COMBINED AND UPDATED ---

def analyze_churn_risk(chat_log):
    system_prompt = """
    You are an expert Customer Success API.
    Analyze customer support chat logs and determine the churn risk as "Low", "Medium", or "High".
    You must reply in valid JSON format with exactly two keys: "risk_level" and "reasoning".
    """

    user_prompt = f"""
    Examples:
    Chat: "How do I reset my password?"
    Output: {{"risk_level": "Low", "reasoning": "Standard technical query, no frustration shown."}}

    Chat: "I've been waiting for 3 days for a reply. This is unacceptable."
    Output: {{"risk_level": "Medium", "reasoning": "Customer is expressing frustration over wait times."}}

    Chat: "Cancel my subscription. Your product is a waste of money."
    Output: {{"risk_level": "High", "reasoning": "Explicit statement of cancellation and severe dissatisfaction."}}

    Now, analyze the following chat:
    Chat: "{chat_log}"
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            # FIXED: Using the active Llama 3.1 model
            model="llama-3.1-8b-instant",
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        response_text = chat_completion.choices[0].message.content
        result = json.loads(response_text)
        return result.get('risk_level', 'Unknown'), result.get('reasoning', 'No reasoning provided')

    except Exception as e:
        print(f"Error: {e}")
        return "Unknown", "Error in LLM pipeline"

# Run the Inference Pipeline
predicted_risks = []
reasonings = []

print("Running Groq Llama-3.1 Inference...")
start_time = time.time()

for index, row in df.iterrows():
    risk, reason = analyze_churn_risk(row['chat_log'])
    predicted_risks.append(risk)
    reasonings.append(reason)
    time.sleep(0.5)

df['llm_predicted_risk'] = predicted_risks
df['llm_reasoning'] = reasonings

end_time = time.time()
print(f"Inference completed in {end_time - start_time:.2f} seconds.\n")

# Display results before evaluation
display(df[['customer_id', 'chat_log', 'human_baseline_risk', 'llm_predicted_risk']])

Running Groq Llama-3.1 Inference...
Inference completed in 3.02 seconds.



,customer_id,chat_log,human_baseline_risk,llm_predicted_risk
0,C-101,"Hi, I'm having trouble exporting my report. It...",Low,Low
1,C-101,This is the second time I'm asking. The export...,Medium,Medium
2,C-101,Your system is completely useless. If this isn...,High,High
3,C-102,Just wanted to say the new UI update looks gre...,Low,Low
4,C-103,Billing error on my account. I was charged twi...,Medium,Medium


In [25]:
# 4. Evaluate the System
eval_df = df[df['llm_predicted_risk'] != "Unknown"]

# Calculate Overall Accuracy
accuracy = accuracy_score(eval_df['human_baseline_risk'], eval_df['llm_predicted_risk'])

# Calculate Cohen's Kappa
kappa = cohen_kappa_score(eval_df['human_baseline_risk'], eval_df['llm_predicted_risk'], labels=["Low", "Medium", "High"])

# Calculate Routing Accuracy
eval_df['human_binary'] = eval_df['human_baseline_risk'].apply(lambda x: 1 if x == 'High' else 0)
eval_df['llm_binary'] = eval_df['llm_predicted_risk'].apply(lambda x: 1 if x == 'High' else 0)
routing_accuracy = accuracy_score(eval_df['human_binary'], eval_df['llm_binary'])

print("\n--- EVALUATION METRICS ---")
print(f"Overall Accuracy:   {accuracy * 100:.1f}%")
print(f"Cohen's Kappa:      {kappa:.3f} (1.0 is perfect agreement)")
print(f"Routing Accuracy:   {routing_accuracy * 100:.1f}%")

if kappa > 0.8:
    print("\nResult: System is ready for A/B testing in production.")
else:
    print("\nResult: Prompt requires further refinement. Kappa score too low.")


--- EVALUATION METRICS ---
Overall Accuracy:   100.0%
Cohen's Kappa:      1.000 (1.0 is perfect agreement)
Routing Accuracy:   100.0%

Result: System is ready for A/B testing in production.
